# 01 - Inspect Existing Network Inputs

**Purpose:** display the transmission routes and substation points supplied by the collaborator, estimate how total demand might be shared between substations, and list the data still needed before outage calculations can run.

**Before running:**

- run `00_data_intake.ipynb` or the equivalent preparation command;
- use the repository `.venv` kernel;
- optionally place GridFinder and OSM line files under `data/0-incoming/energy/`.

The transmission layer is vector data read from `PowerGrid.shp`. It is not traced from the reference PNG. The shapefile shows route geometry, but it does not provide a complete electrical line table with endpoint substations and power ratings. This notebook therefore does not derive model connections from the map.


In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

from mu_star_energy.distribution import build_service_weights
from mu_star_energy.paths import incoming_energy_dir, processed_energy_dir

COLLABORATOR_DIR = processed_energy_dir() / "collaborator"

substations = gpd.read_parquet(COLLABORATOR_DIR / "substations.parquet")
routes = gpd.read_parquet(COLLABORATOR_DIR / "transmission_routes.parquet")


## Supplied transmission mapping

The map below plots the received route geometry and substation points directly. It does not add, split or connect lines.

The accompanying table shows the information available in the route layer. Most records are unnamed, and the layer has no endpoint-substation, circuit-count, line-rating or operating-status fields. A separate image, `network_map_2025.png`, is retained as a visual reference but is not used to build topology.


In [ ]:
route_summary = routes.assign(geometry_type=routes.geometry.geom_type)[
    ["route_id", "name", "voltage_kv_hint", "length_km", "geometry_type"]
]
display(route_summary.rename(columns={
    "route_id": "route ID",
    "name": "source name",
    "voltage_kv_hint": "voltage shown in source (kV)",
    "length_km": "mapped geometry length (km)",
    "geometry_type": "geometry type",
}))

fig, ax = plt.subplots(figsize=(9, 9))
routes.plot(
    ax=ax,
    color="#64748b",
    linewidth=2.2,
    alpha=0.75,
    label="received transmission route",
)
substations.plot(
    ax=ax,
    color="#ff8c00",
    edgecolor="black",
    markersize=50,
    label="received substation point",
)
for _, row in substations.iterrows():
    ax.annotate(
        row["bus_id"],
        (row.geometry.x, row.geometry.y),
        xytext=(3, 3),
        textcoords="offset points",
        fontsize=7,
    )
ax.set_title("Received transmission routes and substation points")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.grid(alpha=0.2)
ax.legend()
plt.show()


## Estimating each substation's share of demand

OSM and GridFinder lines are optional. When available, the code assigns each mapped distribution-line section to the nearest transmission substation. The amount of mapped line near each substation is then used as a rough guide to its share of total demand.

This estimate is only for locating customers and demand. GridFinder routes are estimates and are not added to the electrical network calculation.


In [ ]:
def optional_line_layer(path):
    if not path.exists():
        return None
    return gpd.read_parquet(path) if path.suffix == ".parquet" else gpd.read_file(path)


gridfinder_path = incoming_energy_dir() / "gridfinder" / "grid.gpkg"
osm_path = incoming_energy_dir() / "osm" / "distribution_lines.parquet"
service_weights = build_service_weights(
    substations,
    gridfinder_lines=optional_line_layer(gridfinder_path),
    osm_distribution_lines=optional_line_layer(osm_path),
)
service_weights.to_csv(COLLABORATOR_DIR / "service_weights.csv", index=False)
display(service_weights.rename(columns={
    "bus_id": "substation ID",
    "gridfinder_km": "GridFinder line length (km)",
    "osm_km": "OSM line length (km)",
    "service_weight": "share of total demand",
    "method": "method used",
}))


## Information needed before the model can run

The operational model requires a separate `existing_lines.csv`. Each row must state the line ID, endpoint substations, voltage, length and maximum power. This file is not generated from the mapped route geometry.

For demand over time, the model accepts either one Mauritius-wide `demand_mw` series or one column per substation. Regular half-hourly, hourly and three-hourly data are supported. The current automated workflow only checks that `demand_profile.csv` exists; reading the file and launching outage cases still needs to be added.


In [ ]:
generator_register = pd.read_csv(COLLABORATOR_DIR / "generation_register_template.csv")
existing_lines_path = COLLABORATOR_DIR / "existing_lines.csv"
existing_lines = pd.read_csv(existing_lines_path) if existing_lines_path.exists() else pd.DataFrame()
required_line_columns = {"line_id", "bus0", "bus1", "v_nom_kv", "length_km", "s_nom_mva"}
line_columns_complete = required_line_columns.issubset(existing_lines.columns)
line_values_complete = bool(
    line_columns_complete
    and len(existing_lines)
    and existing_lines[list(required_line_columns)].notna().all().all()
)

readiness = pd.Series({
    "received vector route records": len(routes),
    "existing electrical line table present": int(existing_lines_path.exists()),
    "existing electrical lines listed": len(existing_lines),
    "line endpoint and rating fields complete": int(line_values_complete),
    "power stations with maximum output": int(generator_register["capacity_mw"].notna().sum()),
    "total power stations requiring maximum output": len(generator_register),
    "power stations with running cost": int(generator_register["marginal_cost"].notna().sum()),
    "total power stations requiring running cost": len(generator_register),
    "power stations with connected substation": int(generator_register["bus_id"].notna().sum()),
    "total power stations requiring a substation": len(generator_register),
    "demand-over-time file present": int((COLLABORATOR_DIR / "demand_profile.csv").exists()),
    "method used to share demand": service_weights["method"].iloc[0] if len(service_weights) else "none",
})
display(readiness.to_frame("value"))


## How an outage is described

After the input checks are complete, mu-star supplies a table such as:

| `component` | `asset_id` | `available_fraction` |
|---|---|---:|
| Line | LINE_004 | 0.0 |
| Generator | Fort_George_1 | 0.4 |
| Bus | SUB_008 | 0.0 |

`component` identifies a line, power station (`Generator`) or substation (`Bus`, the PyPSA term). `available_fraction` is the usable share: 1 means fully available, 0 means completely unavailable and 0.4 means 40% available.

`EnergyModel.simulate(network, disruptions)` copies the existing-system model, applies these reductions, recalculates electricity production and reports unmet demand, the share of demand served and operating cost. The separate damage work determines the available fraction supplied to this model.
